#Setup & Configuration

In [2]:
import os
import sys
import json
import time
import traceback
from collections import Counter
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Record lab start time
LAB_START = datetime.now()
print(f"🕐 Lab started at: {LAB_START.strftime('%Y-%m-%d %H:%M:%S')}")

# Load .env and initialize client
load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("❌ OPENAI_API_KEY not found. Check your .env file.")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"
print("✅ Setup complete! OpenAI client initialized.")

🕐 Lab started at: 2026-02-22 18:46:56
✅ Setup complete! OpenAI client initialized.


Helper function

In [ ]:
import traceback

# --- Token limits per task ---
# These act as hard enforcement on top of prompt instructions.
# The prompt says "one word" or "under 100 words" — max_tokens makes it impossible to violate.
MAX_TOKENS_SENTIMENT   = 5    # One word: Positive / Negative / Neutral
MAX_TOKENS_PRODUCT     = 150  # ~100 words described in prompt, slight buffer
MAX_TOKENS_EXTRACTION  = 100  # JSON object only, no trailing commentary

def call_openai(
    prompt: str,
    system_message: str = "You are a helpful assistant.",
    temperature: float = 0.7,
    max_tokens: int | None = None,
    seed: int | None = None
) -> str:
    """Call OpenAI and return response text, or 'ERROR: ...' if it fails."""
    try:
        params = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user",   "content": prompt}
            ],
            "temperature": temperature
        }
        if max_tokens is not None:
            params["max_tokens"] = max_tokens
        if seed is not None:
            params["seed"] = seed

        response = client.chat.completions.create(**params)
        return response.choices[0].message.content

    except Exception as e:
        traceback.print_exc()
        return f"ERROR: {str(e)}"


def display_response(title: str, response: str, params: dict | None = None) -> None:
    """Pretty-print a response with a title and separator lines."""
    print(f"\n{'='*60}")
    print(f"📝 {title}")
    if params:
        print(f"Parameters: {params}")
    print(f"{'='*60}")
    print(response)
    print(f"{'='*60}\n")


print("✅ Helper functions loaded!")

display_response("Lab Setup Complete", "OpenAI client is ready to use. Let's get started with the tasks!")



✅ Helper functions loaded!

📝 Lab Setup Complete
OpenAI client is ready to use. Let's get started with the tasks!



True

Task 1: Sentiment Analysis

In [8]:
# ============================================================
# Cell 3 — Task 1: Sentiment Analysis v1 (Zero-Shot Baseline)
# ============================================================

# This is the simplest possible prompt — no instructions, no format requirements.
# We're establishing a baseline to see what the model does when left to its own devices.
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

result = call_openai(sentiment_prompt_v1)
print("Sentiment Analysis v1 Result:")
print(result)


Sentiment Analysis v1 Result:
This customer message can be classified as **positive feedback** or **customer satisfaction**.


Product description generation

In [9]:
# ============================================================
# Cell 4 — Task 2: Product Description v1 (Zero-Shot Baseline)
# ============================================================

# No structure, no length constraints, no style guidelines.
# The model will produce something reasonable but completely inconsistent across runs.
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

result = call_openai(product_prompt_v1)
print("Product Description v1 Result:")
print(result)

Product Description v1 Result:
**Product Description: Wireless Precision Mouse**

Elevate your computing experience with our Wireless Precision Mouse, designed for seamless navigation and unparalleled comfort. Priced at just $29.99, this sleek and modern accessory combines cutting-edge technology with ergonomic design, making it the perfect companion for both work and play.

**Key Features:**

- **Wireless Freedom:** Enjoy the convenience of a clutter-free workspace with advanced 2.4GHz wireless technology. Say goodbye to tangled cords and hello to effortless movement!

- **Ergonomic Comfort:** Crafted with your comfort in mind, the Wireless Precision Mouse features a contoured shape that fits perfectly in your hand, reducing strain during long hours of use.

- **High Precision Tracking:** Equipped with a high-resolution optical sensor, this mouse delivers smooth and accurate tracking on various surfaces, ensuring precision in every click and scroll.

- **Customizable Buttons:** Person

Task 3: Data Extraction

In [10]:
# ============================================================
# Cell 5 — Task 3: Data Extraction v1 (Zero-Shot Baseline)
# ============================================================

# No output format specified — the model will extract data but present it however it wants.
# Watch how it chooses bullet points over structured JSON without being told to.
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. 
The delivery was fast but the packaging was damaged."
"""

result = call_openai(extraction_prompt_v1)
print("Data Extraction v1 Result:")
print(result)

Data Extraction v1 Result:
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged


Part 2: Diagnosing Failures - Systematic Testing

Step 3: Run Prompts 5 Times

Prompt = "I love this product! It's exactly what I needed."

In [11]:
# ============================================================
# Cell 6 — 5-Run Consistency Test: Sentiment Analysis v1
# ============================================================

# WHY: Running the same prompt 5 times reveals that even identical inputs
# produce different outputs due to temperature-based randomness.
# This is our first evidence that prompt engineering is empirical, not theoretical.

prompt = "I love this product! It's exactly what I needed."
temperature = 0.7
num_runs = 5

print(f"Running sentiment prompt {num_runs} times at temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    
    # Skip and flag error responses so they don't corrupt our analysis
    if is_error_response(response):
        print(f"\n🔄 Run #{i+1}: ❌ API Error — {response}")
        con

Running sentiment prompt 5 times at temperature=0.7
Prompt: 'I love this product! It's exactly what I needed.'



Prompt = "Create a product description for a wireless mouse that costs $29.99."

In [12]:
# ============================================================
# Cell 7 — 5-Run Consistency Test: Product Description v1
# ============================================================

# WHY: Generation tasks show the most dramatic inconsistency at this stage.
# Watch how the product name, structure, and length all change every run.

prompt = "Create a product description for a wireless mouse that costs $29.99."
temperature = 0.7
num_runs = 5

print(f"Running product description prompt {num_runs} times at temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    
    if is_error_response(response):
        print(f"\n🔄 Run #{i+1}: ❌ API Error — {response}")
        continue
        
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-" * 80)
    time.sleep(0.5)

print("\n" + "=" * 80)
print("📊 Analysis:")
print(f"Successful runs: {len(responses)} out of {num_runs}")
print(f"Unique responses: {len(set(responses))} out of {len(responses)}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running product description prompt 5 times at temperature=0.7
Prompt: 'Create a product description for a wireless mouse that costs $29.99.'


🔄 Run #1:
**Product Description: Wireless Ergonomic Mouse - $29.99**

Upgrade your workspace with our Wireless Ergonomic Mouse, designed for comfort and efficiency. Priced at just $29.99, this sleek and stylish mouse offers a perfect blend of functionality and modern design, making it the ideal companion for both home and office use.

**Key Features:**

- **Ergonomic Design:** Our mouse is shaped to fit comfortably in your hand, reducing strain during long hours of use. The contoured design promotes a natural grip, ensuring your wrist stays relaxed and supported.

- **Wireless Convenience:** Say goodbye to tangled cords! The wireless connectivity provides you with the freedom to move seamlessly across your desk, while the reliable 2.4GHz technology ensures a stable and interference-free connection.

- **Precision Tracking:** Equipped with a high

Prompt = "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."

In [13]:
# ============================================================
# Cell 8 — 5-Run Consistency Test: Data Extraction v1
# ============================================================

# WHY: Extraction tasks fail differently from classification and generation.
# The data extracted is correct but the FORMAT is completely unpredictable —
# sometimes bullet points, sometimes prose, sometimes JSON, sometimes a table.

prompt = "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
temperature = 0.7
num_runs = 5

print(f"Running data extraction prompt {num_runs} times at temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    
    if is_error_response(response):
        print(f"\n🔄 Run #{i+1}: ❌ API Error — {response}")
        continue
        
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-" * 80)
    time.sleep(0.5)

print("\n" + "=" * 80)
print("📊 Analysis:")
print(f"Successful runs: {len(responses)} out of {num_runs}")
print(f"Unique responses: {len(set(responses))} out of {len(responses)}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running data extraction prompt 5 times at temperature=0.7
Prompt: 'I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.'


🔄 Run #1:
I'm sorry to hear that your packaging was damaged upon delivery. It's great to know that the delivery was fast, but the condition of the packaging is important as well. If you need assistance with this issue, I recommend contacting the customer service of the company you ordered from. They can help you with potential returns, replacements, or any other concerns you may have regarding the damaged packaging. Would you like any specific advice on how to proceed?
--------------------------------------------------------------------------------

🔄 Run #2:
I'm sorry to hear that the packaging was damaged when your order arrived. It's great that the delivery was fast, but packaging issues can be concerning. If you haven't already, I recommend reaching out to the seller or customer service to report the damaged packaging. They 

Step 4: Run Prompts 10 Times

In [14]:
# ============================================================
# Cell 9 — 10-Run Consistency Test: Sentiment Analysis v1
# ============================================================

# WHY: 5 runs gave us a glimpse of inconsistency.
# 10 runs starts revealing PATTERNS — do certain response styles
# repeat? Does inconsistency get worse or stabilize?

prompt = "I love this product! It's exactly what I needed."
temperature = 0.7
num_runs = 10

print(f"Running sentiment prompt {num_runs} times at temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    
    if is_error_response(response):
        print(f"\n🔄 Run #{i+1}: ❌ API Error — {response}")
        continue
        
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-" * 80)
    time.sleep(0.5)

# Count how many times each unique response appeared
response_counts = Counter(responses)

print("\n" + "=" * 80)
print("📊 Analysis:")
print(f"Successful runs: {len(responses)} out of {num_runs}")
print(f"Unique responses: {len(set(responses))} out of {len(responses)}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print(f"\n📋 Repeated responses:")
for response, count in response_counts.most_common():
    if count > 1:
        print(f"  x{count}: '{response[:60]}...'")
    else:
        print(f"  x1 (unique): '{response[:60]}...'")

Running sentiment prompt 10 times at temperature=0.7
Prompt: 'I love this product! It's exactly what I needed.'


🔄 Run #1:
I'm glad to hear that you love the product! What specifically do you like about it?
--------------------------------------------------------------------------------

🔄 Run #2:
That's great to hear! I'm glad you found a product that meets your needs. What specifically do you love about it?
--------------------------------------------------------------------------------

🔄 Run #3:
That's great to hear! I'm glad you found a product that meets your needs. What specifically do you love about it?
--------------------------------------------------------------------------------

🔄 Run #4:
I'm glad to hear that you love the product! What specific features or aspects do you find most helpful?
--------------------------------------------------------------------------------

🔄 Run #5:
I'm glad to hear that you love the product! What do you like most about it?
----------------

In [ ]:
# ============================================================
# Cell 10 — 10-Run Consistency Test: Product Description v1
# ============================================================

# WHY: At 10 runs, generation tasks show something interesting —
# the model keeps inventing NEW product names every time.
# There's no convergence at all, unlike classification tasks.
# This tells us generation needs structural constraints, not just more runs.

prompt = "Create a product description for a wireless mouse that costs $29.99."
temperature = 0.7
num_runs = 10

print(f"Running product description prompt {num_runs} times at temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    
    if is_error_response(response):
        print(f"\n🔄 Run #{i+1}: ❌ API Error — {response}")
        continue
        
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-" * 80)
    time.sleep(0.5)

response_counts = Counter(responses)

print("\n" + "=" * 80)
print("📊 Analysis:")
print(f"Successful runs: {len(responses)} out of {num_runs}")
print(f"Unique responses: {len(set(responses))} out of {len(responses)}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print(f"\n📋 Repeated responses:")
for response, count in response_counts.most_common():
    if count > 1:
        print(f"  x{count}: '{response[:60]}...'")
    else:
        print(f"  x1 (unique): '{response[:60]}...'")

Running the same prompt 10 times with temperature=0.7
Prompt: 'Create a product description for a wireless mouse that costs $29.99.'


🔄 Run #1:
**Product Description: Wireless Freedom: Ergonomic Wireless Mouse - $29.99**

Elevate your computing experience with our Ergonomic Wireless Mouse, designed for comfort, precision, and convenience. Priced at just $29.99, this sleek accessory combines functionality and style, making it the perfect companion for both work and play.

**Key Features:**

- **Wireless Convenience:** Say goodbye to tangled cords! Our wireless mouse offers a seamless connection with a reliable USB receiver, allowing you to enjoy the freedom of movement without compromising on performance.

- **Ergonomic Design:** Crafted with your comfort in mind, the ergonomic shape fits naturally in your hand, reducing wrist strain during long hours of use. The textured grip ensures stability, making it ideal for both casual browsing and intensive gaming sessions.

- **High Precision

In [15]:
# ============================================================
# Cell 11 — 10-Run Consistency Test: Data Extraction v1
# ============================================================

# WHY: At 10 runs, extraction shows a consistent TASK CONFUSION pattern —
# the model keeps responding to the feedback as a customer service agent
# instead of extracting structured data.
# This is a prompt design failure, not a temperature problem.

prompt = "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
temperature = 0.7
num_runs = 10

print(f"Running data extraction prompt {num_runs} times at temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    
    if is_error_response(response):
        print(f"\n🔄 Run #{i+1}: ❌ API Error — {response}")
        continue
        
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-" * 80)
    time.sleep(0.5)

response_counts = Counter(responses)

print("\n" + "=" * 80)
print("📊 Analysis:")
print(f"Successful runs: {len(responses)} out of {num_runs}")
print(f"Unique responses: {len(set(responses))} out of {len(responses)}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print(f"\n📋 Repeated responses:")
for response, count in response_counts.most_common():
    if count > 1:
        print(f"  x{count}: '{response[:60]}...'")
    else:
        print(f"  x1 (unique): '{response[:60]}...'")

Running data extraction prompt 10 times at temperature=0.7
Prompt: 'I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.'


🔄 Run #1:
I'm glad to hear that your delivery was fast, but I'm sorry to hear that the packaging was damaged. If you need assistance with this issue, you might want to contact customer service for the retailer you ordered from. They can help you with options such as a replacement, return, or refund. Make sure to have your order number (item #12345) handy when you reach out to them.
--------------------------------------------------------------------------------

🔄 Run #2:
I'm glad to hear that your delivery was fast, but I'm sorry to hear that the packaging was damaged. If you would like, I can help you with steps to report the issue or assist you in getting a replacement or refund. Please let me know how you would like to proceed!
--------------------------------------------------------------------------------

🔄 Run #3:
I'm s

Step 5: Run Prompts 15 Times 

Prompt: "I love this product! It's exactly what I needed."

In [16]:
# ============================================================
# Cell 12 — 15-Run Failure Analysis: Sentiment Analysis v1
# ============================================================

# WHY: 15 runs is our definitive baseline measurement.
# We now build a proper failure analysis DataFrame so we can
# compare v1 vs v2 vs v3 consistency with real numbers.
# 
# IMPORTANT: This is the CORRECT prompt for sentiment analysis.
# The system_message here is neutral — we are NOT injecting
# extraction instructions into a sentiment task.

prompt = "I love this product! It's exactly what I needed."
system_message = "You are a helpful assistant."
temperature = 0.7
num_runs = 15

responses_data = []

print(f"🚀 Starting 15-run test: Sentiment Analysis v1")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

for i in range(num_runs):
    print(f"Executing Run {i+1}/{num_runs}...")
    response = call_openai(prompt, system_message=system_message, temperature=temperature)
    
    # Check for API errors first — don't evaluate broken responses
    if is_error_response(response):
        print(f"❌ API Error: {response}")
        responses_data.append({
            "run_id": i + 1,
            "prompt": prompt,
            "response": response,
            "is_success": False,
            "failure_pattern": "API Error"
        })
        continue
    
    # A valid sentiment response should be a single word from our target labels.
    # Any response longer than 20 characters is almost certainly not a clean label.
    valid_labels = ["positive", "negative", "neutral"]
    is_valid = response.strip().lower() in valid_labels
    failure_pattern = "None (Success)" if is_valid else "Wrong Format — not a clean label"
    
    responses_data.append({
        "run_id": i + 1,
        "prompt": prompt,
        "response": response,
        "is_success": is_valid,
        "failure_pattern": failure_pattern
    })
    
    print(f"Response: '{response[:80]}'")
    print(f"Valid: {'✅' if is_valid else '❌'} — {failure_pattern}")
    print("-" * 40)
    time.sleep(0.5)

# Build summary DataFrame
df_sentiment_v1 = pd.DataFrame(responses_data)

print("\n" + "=" * 60)
print("📊 SUMMARY: Sentiment Analysis v1")
print("=" * 60)
success_rate = df_sentiment_v1['is_success'].sum() / len(df_sentiment_v1) * 100
print(f"Total runs:   {len(df_sentiment_v1)}")
print(f"Successes:    {df_sentiment_v1['is_success'].sum()}")
print(f"Failures:     {len(df_sentiment_v1) - df_sentiment_v1['is_success'].sum()}")
print(f"Success Rate: {success_rate:.1f}%")
print(f"\n📋 Failure Patterns:")
print(df_sentiment_v1['failure_pattern'].value_counts().to_string())

🚀 Starting 15-run test: Sentiment Analysis v1
Prompt: 'I love this product! It's exactly what I needed.'

Executing Run 1/15...
Response: 'That's great to hear! I'm glad you found a product that meets your needs. What d'
Valid: ❌ — Wrong Format — not a clean label
----------------------------------------
Executing Run 2/15...
Response: 'That's great to hear! It sounds like you've found something that really meets yo'
Valid: ❌ — Wrong Format — not a clean label
----------------------------------------
Executing Run 3/15...
Response: 'I'm glad to hear that you love the product! What specifically do you like about '
Valid: ❌ — Wrong Format — not a clean label
----------------------------------------
Executing Run 4/15...
Response: 'That's great to hear! I'm glad you found a product that meets your needs. What d'
Valid: ❌ — Wrong Format — not a clean label
----------------------------------------
Executing Run 5/15...
Response: 'I'm glad to hear that you love the product! What specifically

In [21]:
# ============================================================
# Cell 13 — 15-Run Failure Analysis: Product Description v1
# ============================================================

# WHY: For generation tasks, binary success/fail is a misleading metric.
# The real failures are in consistency — word count, structure, and invented content.
# We track all three here to establish a meaningful baseline to beat in v2 and v3.

prompt = "Create a product description for a wireless mouse that costs $29.99."
system_message = "You are a helpful assistant."
temperature = 0.7
num_runs = 15
TARGET_WORD_COUNT = 100  # Our v2 target — used here to show how far v1 is from it

responses_data = []

print(f"🚀 Starting 15-run test: Product Description v1")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

for i in range(num_runs):
    print(f"Executing Run {i+1}/{num_runs}...")
    response = call_openai(prompt, system_message=system_message, temperature=temperature)
    
    if is_error_response(response):
        print(f"❌ API Error: {response}")
        responses_data.append({
            "run_id": i + 1,
            "prompt": prompt,
            "response": response,
            "is_success": False,
            "word_count": 0,
            "within_word_limit": False,
            "has_headline": False,
            "has_bullets": False,
            "failure_pattern": "API Error"
        })
        continue
    
    word_count = len(response.split())
    
    # Check 1: Is it non-empty? (low bar — almost always passes in v1)
    is_non_empty = len(response) > 50
    
    # Check 2: Is it within the 100-word target?
    # v1 will almost always fail this — that's the point
    within_word_limit = word_count <= TARGET_WORD_COUNT
    
    # Check 3: Does it have a headline?
    # We look for ** markdown bold which the model uses for titles
    has_headline = "**" in response
    
    # Check 4: Does it have bullet points?
    has_bullets = "- " in response or "• " in response
    
    # True success requires ALL checks to pass
    # In v1 this will rarely happen — surfacing the real failure rate
    is_valid = is_non_empty and within_word_limit and has_headline and has_bullets
    
    # Build a descriptive failure pattern
    if not is_non_empty:
        failure_pattern = "Empty response"
    elif not within_word_limit:
        failure_pattern = f"Over word limit ({word_count} words vs {TARGET_WORD_COUNT} target)"
    elif not has_headline:
        failure_pattern = "Missing headline"
    elif not has_bullets:
        failure_pattern = "Missing bullet points"
    else:
        failure_pattern = "None (Success)"
    
    responses_data.append({
        "run_id": i + 1,
        "prompt": prompt,
        "response": response,
        "is_success": is_valid,
        "word_count": word_count,
        "within_word_limit": within_word_limit,
        "has_headline": has_headline,
        "has_bullets": has_bullets,
        "failure_pattern": failure_pattern
    })
    
    print(f"Response: '{response[:80]}...'")
    print(f"Word count: {word_count} | Within limit: {'✅' if within_word_limit else '❌'} | Headline: {'✅' if has_headline else '❌'} | Bullets: {'✅' if has_bullets else '❌'}")
    print(f"Valid: {'✅' if is_valid else '❌'} — {failure_pattern}")
    print("-" * 40)
    time.sleep(0.5)

df_product_v1 = pd.DataFrame(responses_data)

print("\n" + "=" * 60)
print("SUMMARY: Product Description v1")
print("=" * 60)
success_rate = df_product_v1['is_success'].sum() / len(df_product_v1) * 100
print(f"Total runs:        {len(df_product_v1)}")
print(f"Successes:         {df_product_v1['is_success'].sum()}")
print(f"Failures:          {len(df_product_v1) - df_product_v1['is_success'].sum()}")
print(f"Success Rate:      {success_rate:.1f}%")
print(f"\n📏 Word Count Analysis:")
print(f"Average:           {df_product_v1['word_count'].mean():.0f} words")
print(f"Min:               {df_product_v1['word_count'].min()} words")
print(f"Max:               {df_product_v1['word_count'].max()} words")
print(f"Target:            {TARGET_WORD_COUNT} words")
print(f"Within limit:      {df_product_v1['within_word_limit'].sum()}/15 runs")
print(f"\n📋 Structure Analysis:")
print(f"Has headline:      {df_product_v1['has_headline'].sum()}/15 runs")
print(f"Has bullets:       {df_product_v1['has_bullets'].sum()}/15 runs")
print(f"\n📋 Failure Patterns:")
print(df_product_v1['failure_pattern'].value_counts().to_string())

🚀 Starting 15-run test: Product Description v1
Prompt: 'Create a product description for a wireless mouse that costs $29.99.'

Executing Run 1/15...
Response: '**Product Description: Wireless Precision Mouse**

Elevate your computing experi...'
Word count: 287 | Within limit: ❌ | Headline: ✅ | Bullets: ✅
Valid: ❌ — Over word limit (287 words vs 100 target)
----------------------------------------
Executing Run 2/15...
Response: '**Product Description: Wireless Comfort Mouse - $29.99**

Upgrade your workspace...'
Word count: 338 | Within limit: ❌ | Headline: ✅ | Bullets: ✅
Valid: ❌ — Over word limit (338 words vs 100 target)
----------------------------------------
Executing Run 3/15...
Response: '**Product Description: Wireless Precision Mouse**

Elevate your workspace with o...'
Word count: 267 | Within limit: ❌ | Headline: ✅ | Bullets: ✅
Valid: ❌ — Over word limit (267 words vs 100 target)
----------------------------------------
Executing Run 4/15...
Response: '**Product Description

In [43]:
# ============================================================
# Cell 14 — 15-Run Failure Analysis: Data Extraction v1
# ============================================================

# WHY: This is the only task where v1 uses a system message that makes sense.
# But it has a hidden failure — the model wraps JSON in markdown blocks
# which makes json.loads() fail in a real pipeline without cleaning.
#
# KEY: We do NOT strip markdown before parsing here.
# That's intentional — we want v1 to correctly fail on markdown wrappers
# so the improvement in v2 and v3 is visible and meaningful.

prompt = "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
system_message = "Extract the following information and return as JSON: order_id, order_date, delivery_speed, packaging_condition."
temperature = 0.7
num_runs = 15

responses_data = []

print("Starting 15-run test: Data Extraction v1")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

for i in range(num_runs):
    print(f"Executing Run {i+1}/{num_runs}...")
    response = call_openai(
        prompt,
        system_message=system_message,
        temperature=temperature
    )

    if is_error_response(response):
        print(f"API Error: {response}")
        responses_data.append({
            "run_id": i + 1,
            "response": response,
            "is_success": False,
            "has_fields": False,
            "is_parseable": False,
            "failure_pattern": "API Error"
        })
        continue

    # Check 1: Do all required fields exist in the response?
    required_fields = ['order_id', 'order_date', 'delivery_speed', 'packaging_condition']
    has_fields = all(field in response.lower() for field in required_fields)

    # Check 2: Is the raw response directly parseable as JSON?
    # We intentionally do NOT strip markdown here.
    # If the model wrapped it in backtick json blocks it should fail.
    # That is the v1 failure pattern we are measuring.
    try:
        json.loads(response.strip())
        is_parseable = True
    except json.JSONDecodeError:
        is_parseable = False

    is_valid = has_fields and is_parseable

    if not has_fields:
        failure_pattern = "Missing Fields"
    elif not is_parseable:
        failure_pattern = "Markdown Wrapper - not clean JSON"
    else:
        failure_pattern = "None (Success)"

    responses_data.append({
        "run_id": i + 1,
        "response": response,
        "is_success": is_valid,
        "has_fields": has_fields,
        "is_parseable": is_parseable,
        "failure_pattern": failure_pattern
    })

    print(f"Response: '{response[:80]}...'")
    print(f"Has fields: {'yes' if has_fields else 'no'} | Parseable: {'yes' if is_parseable else 'no'}")
    print(f"Valid: {'yes' if is_valid else 'no'} -- {failure_pattern}")
    print("-" * 40)
    time.sleep(0.5)

df_extraction_v1 = pd.DataFrame(responses_data)

print("\n" + "=" * 60)
print("SUMMARY: Data Extraction v1")
print("=" * 60)
success_rate = df_extraction_v1['is_success'].sum() / len(df_extraction_v1) * 100
print(f"Total runs:   {len(df_extraction_v1)}")
print(f"Successes:    {df_extraction_v1['is_success'].sum()}")
print(f"Failures:     {len(df_extraction_v1) - df_extraction_v1['is_success'].sum()}")
print(f"Success Rate: {success_rate:.1f}%")
print(f"\nFailure Patterns:")
print(df_extraction_v1['failure_pattern'].value_counts().to_string())

Starting 15-run test: Data Extraction v1
Prompt: 'I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.'

Executing Run 1/15...
Response: '```json
{
  "order_id": "12345",
  "order_date": "2023-03-15",
  "delivery_speed...'
Has fields: yes | Parseable: no
Valid: no -- Markdown Wrapper - not clean JSON
----------------------------------------
Executing Run 2/15...
Response: '```json
{
  "order_id": "12345",
  "order_date": "2023-03-15",
  "delivery_speed...'
Has fields: yes | Parseable: no
Valid: no -- Markdown Wrapper - not clean JSON
----------------------------------------
Executing Run 3/15...
Response: '```json
{
  "order_id": "12345",
  "order_date": "2023-03-15",
  "delivery_speed...'
Has fields: yes | Parseable: no
Valid: no -- Markdown Wrapper - not clean JSON
----------------------------------------
Executing Run 4/15...
Response: '```json
{
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_speed...'
Has fields: yes | Parseabl

Part 3: Iteration 1 - Rewriting Simple Prompts

In [25]:
# ============================================================
# Cell 15 — Task 1: Sentiment Analysis v2 (Structured Prompt)
# ============================================================

# WHAT CHANGED FROM v1:
# 1. System message now defines the correct role — classification engine, not helpful assistant
# 2. Prompt explicitly lists the allowed labels
# 3. Single word constraint added
# 4. max_tokens=MAX_TOKENS_SENTIMENT enforces the one-word limit at API level

sentiment_prompt_v2 = """
Task: Classify the sentiment of the provided customer message.

Instructions:
- Analyze the tone and intent of the text.
- Use exactly one word for the classification.
- Choose from these labels: [Positive, Neutral, Negative].

Output Format: Provide only the label as a single word. No punctuation or explanation.

Customer Message: "I love this product! It's exactly what I needed."
"""

# System message now aligned with the task — no longer "helpful assistant"
result = call_openai(
    sentiment_prompt_v2,
    system_message="You are a sentiment classification engine. You only respond with a single word: Positive, Negative, or Neutral. Nothing else.",
    max_tokens=MAX_TOKENS_SENTIMENT
)

print("Sentiment Analysis v2 Result:")
print(result)

Sentiment Analysis v2 Result:
Positive


Step 7: Improve Product Description Prompt

In [26]:
# ============================================================
# Cell 16 — Task 2: Product Description v2 (Structured Prompt)
# ============================================================

# WHAT CHANGED FROM v1:
# 1. System message defines the correct role — copywriter, not helpful assistant
# 2. Explicit structure requirements — headline, value proposition, 3 bullets
# 3. Length constraint — under 100 words
# 4. Target audience defined — students and remote workers
# 5. max_tokens=MAX_TOKENS_PRODUCT enforces length at API level

product_prompt_v2 = """
Task: Write a compelling product description for a $29.99 wireless mouse.

Structure Requirements:
1. Catchy Headline: A short, punchy title.
2. Value Proposition: A 2-sentence paragraph focusing on comfort and value.
3. Feature Bullets: Exactly 3 bullet points highlighting technical specs 
   (e.g., battery life, DPI, connectivity).

Style Guidelines:
- Tone: Professional yet approachable.
- Target Audience: Students and remote office workers.
- Length: Total word count must be under 100 words.

Constraints:
- Do not mention specific competitor brand names.
- Focus on ergonomic feel as a primary selling point.
"""

result = call_openai(
    product_prompt_v2,
    system_message="You are a professional product copywriter. You follow structural requirements exactly and never exceed word limits.",
    max_tokens=MAX_TOKENS_PRODUCT
)

print("Product Description v2 Result:")
print(result)

Product Description v2 Result:
### GlidePro Wireless Mouse: Comfort Meets Precision

Experience unparalleled comfort and seamless connectivity with the GlidePro Wireless Mouse, designed for students and remote workers alike. Enjoy exceptional value without sacrificing performance, making every task effortless.

- **Ergonomic Design**: Contoured shape reduces wrist strain for hours of comfortable use.
- **Long Battery Life**: Lasts up to 12 months on a single AA battery, ensuring uninterrupted productivity.
- **High Precision**: Adjustable DPI settings up to 2400 for smooth tracking on any surface.


Step 8: Improve Data Extraction Prompt

In [27]:
# ============================================================
# Cell 17 — Task 3: Data Extraction v2 (Structured Prompt)
# ============================================================

# WHAT CHANGED FROM v1:
# 1. System message defines extraction role — not helpful assistant
# 2. Explicit fields to extract with clear names
# 3. Date format standardised to YYYY-MM-DD
# 4. Output format explicitly says no markdown blocks
# 5. max_tokens=MAX_TOKENS_EXTRACTION prevents trailing commentary

extraction_prompt_v2 = """
Task: Extract specific data points from the provided customer feedback 
into a structured format.

Fields to Extract:
1. order_id: The numerical identifier for the purchase.
2. order_date: The date of the order (formatted as YYYY-MM-DD).
3. sentiment: Overall sentiment (Positive/Neutral/Negative).
4. reported_issues: List any problems mentioned.

Output Format:
Return the data as a clean JSON object. 
Do not include any introductory text or markdown blocks like ```json.

Customer Feedback: "I ordered item #12345 on March 15th. 
The delivery was fast but the packaging was damaged."
"""

result = call_openai(
    extraction_prompt_v2,
    system_message="You are a data extraction engine. You only return clean JSON. No markdown, no explanation, no commentary.",
    max_tokens=MAX_TOKENS_EXTRACTION
)

print("Data Extraction v2 Result:")
print(result)

Data Extraction v2 Result:
{
  "order_id": 12345,
  "order_date": "2023-03-15",
  "sentiment": "Neutral",
  "reported_issues": [
    "damaged packaging"
  ]
}


In [28]:
# ============================================================
# Cell 18 — 15-Run Consistency Test: Sentiment Analysis v2
# ============================================================

# WHY: We now run the same 15-run test on v2 to measure improvement.
# The key metric to watch is success rate — v1 was ~0%, v2 should be much higher.
# We use the same validation logic as Cell 12 so the comparison is fair.

sentiment_prompt_v2 = """
Task: Classify the sentiment of the provided customer message.

Instructions:
- Analyze the tone and intent of the text.
- Use exactly one word for the classification.
- Choose from these labels: [Positive, Neutral, Negative].

Output Format: Provide only the label as a single word. No punctuation or explanation.

Customer Message: "I love this product! It's exactly what I needed."
"""

system_message = "You are a sentiment classification engine. You only respond with a single word: Positive, Negative, or Neutral. Nothing else."
temperature = 0.7
num_runs = 15

responses_data = []

print("Starting 15-run test: Sentiment Analysis v2")
print(f"Prompt version: v2 (structured with explicit labels)")
print(f"System message: '{system_message}'\n")
print("=" * 80)

for i in range(num_runs):
    print(f"Executing Run {i+1}/{num_runs}...")
    response = call_openai(
        sentiment_prompt_v2,
        system_message=system_message,
        temperature=temperature,
        max_tokens=MAX_TOKENS_SENTIMENT
    )

    if is_error_response(response):
        print(f"API Error: {response}")
        responses_data.append({
            "run_id": i + 1,
            "response": response,
            "is_success": False,
            "failure_pattern": "API Error"
        })
        continue

    valid_labels = ["positive", "negative", "neutral"]
    is_valid = response.strip().lower() in valid_labels
    failure_pattern = "None (Success)" if is_valid else "Wrong Format"

    responses_data.append({
        "run_id": i + 1,
        "response": response,
        "is_success": is_valid,
        "failure_pattern": failure_pattern
    })

    print(f"Response: '{response}'")
    print(f"Valid: {'yes' if is_valid else 'no'} -- {failure_pattern}")
    print("-" * 40)
    time.sleep(0.5)

df_sentiment_v2 = pd.DataFrame(responses_data)

# Compare v1 vs v2 side by side
v1_rate = df_sentiment_v1['is_success'].sum() / len(df_sentiment_v1) * 100
v2_rate = df_sentiment_v2['is_success'].sum() / len(df_sentiment_v2) * 100

print("\n" + "=" * 60)
print("COMPARISON: Sentiment Analysis v1 vs v2")
print("=" * 60)
print(f"v1 Success Rate: {v1_rate:.1f}%")
print(f"v2 Success Rate: {v2_rate:.1f}%")
print(f"Improvement:     +{v2_rate - v1_rate:.1f}%")
print(f"\nFailure Patterns v2:")
print(df_sentiment_v2['failure_pattern'].value_counts().to_string())

Starting 15-run test: Sentiment Analysis v2
Prompt version: v2 (structured with explicit labels)
System message: 'You are a sentiment classification engine. You only respond with a single word: Positive, Negative, or Neutral. Nothing else.'

Executing Run 1/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 2/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 3/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 4/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 5/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 6/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 7/15...
Response: 'Positive'
Valid: yes -- None (Success)
--------

In [35]:
# ============================================================
# Cell 19 — 15-Run Consistency Test: Product Description v2
# ============================================================

# WHY: We measure the same four checks as Cell 13 so the comparison is fair.
# The key metric to watch is word count — v1 averaged 282 words,
# v2 should be consistently under 100.

product_prompt_v2 = """
Task: Write a compelling product description for a $29.99 wireless mouse.

Structure Requirements:
1. Catchy Headline: A short, punchy title.
2. Value Proposition: A 2-sentence paragraph focusing on comfort and value.
3. Feature Bullets: Exactly 3 bullet points highlighting technical specs
   (e.g., battery life, DPI, connectivity).

Style Guidelines:
- Tone: Professional yet approachable.
- Target Audience: Students and remote office workers.
- Length: Total word count must be under 100 words.

Constraints:
- Do not mention specific competitor brand names.
- Focus on ergonomic feel as a primary selling point.
"""

system_message = "You are a professional product copywriter. You follow structural requirements exactly and never exceed word limits."
temperature = 0.7
num_runs = 15
TARGET_WORD_COUNT = 100

responses_data = []

print("Starting 15-run test: Product Description v2")
print(f"System message: '{system_message}'\n")
print("=" * 80)

for i in range(num_runs):
    print(f"Executing Run {i+1}/{num_runs}...")
    response = call_openai(
        product_prompt_v2,
        system_message=system_message,
        temperature=temperature,
        max_tokens=MAX_TOKENS_PRODUCT
    )

    if is_error_response(response):
        print(f"API Error: {response}")
        responses_data.append({
            "run_id": i + 1,
            "response": response,
            "is_success": False,
            "word_count": 0,
            "within_word_limit": False,
            "has_headline": False,
            "has_bullets": False,
            "failure_pattern": "API Error"
        })
        continue

    word_count = len(response.split())
    within_word_limit = word_count <= TARGET_WORD_COUNT
    has_headline = "**" in response
    has_bullets = "- " in response or "- " in response
    is_valid = within_word_limit and has_headline and has_bullets

    if not within_word_limit:
        failure_pattern = f"Over word limit ({word_count} words vs {TARGET_WORD_COUNT} target)"
    elif not has_headline:
        failure_pattern = "Missing headline"
    elif not has_bullets:
        failure_pattern = "Missing bullet points"
    else:
        failure_pattern = "None (Success)"

    responses_data.append({
        "run_id": i + 1,
        "response": response,
        "is_success": is_valid,
        "word_count": word_count,
        "within_word_limit": within_word_limit,
        "has_headline": has_headline,
        "has_bullets": has_bullets,
        "failure_pattern": failure_pattern
    })

    print(f"Response: '{response[:80]}...'")
    print(f"Word count: {word_count} | Within limit: {'yes' if within_word_limit else 'no'} | Headline: {'yes' if has_headline else 'no'} | Bullets: {'yes' if has_bullets else 'no'}")
    print(f"Valid: {'yes' if is_valid else 'no'} -- {failure_pattern}")
    print("-" * 40)
    time.sleep(0.5)

df_product_v2 = pd.DataFrame(responses_data)

# Compare v1 vs v2
v1_rate = df_product_v1['is_success'].sum() / len(df_product_v1) * 100
v2_rate = df_product_v2['is_success'].sum() / len(df_product_v2) * 100
v1_avg_words = df_product_v1['word_count'].mean()
v2_avg_words = df_product_v2['word_count'].mean()

print("\n" + "=" * 60)
print("COMPARISON: Product Description v1 vs v2")
print("=" * 60)
print(f"v1 Success Rate:   {v1_rate:.1f}% | Avg words: {v1_avg_words:.0f}")
print(f"v2 Success Rate:   {v2_rate:.1f}% | Avg words: {v2_avg_words:.0f}")
print(f"Improvement:       +{v2_rate - v1_rate:.1f}% | -{v1_avg_words - v2_avg_words:.0f} words avg")
print(f"\nFailure Patterns v2:")
print(df_product_v2['failure_pattern'].value_counts().to_string())

Starting 15-run test: Product Description v2
System message: 'You are a professional product copywriter. You follow structural requirements exactly and never exceed word limits.'

Executing Run 1/15...
Response: '### Elevate Your Productivity with Comfort

Experience seamless navigation with ...'
Word count: 79 | Within limit: yes | Headline: yes | Bullets: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 2/15...
Response: '### GlidePro Wireless Mouse: Comfort Meets Performance

Discover unparalleled co...'
Word count: 81 | Within limit: yes | Headline: yes | Bullets: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 3/15...
Response: '**Effortless Precision: Your Ultimate Wireless Companion**

Experience unparalle...'
Word count: 77 | Within limit: yes | Headline: yes | Bullets: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 4/15...
Response: '### GlidePro Wireless Mou

In [30]:
# ============================================================
# Cell 20 — 15-Run Consistency Test: Data Extraction v2
# ============================================================

# WHY: The key improvement in v2 for extraction is clean JSON — no markdown wrappers.
# v1 had ~53% clean JSON. v2 explicitly forbids markdown in both the
# prompt and system message. We measure if that instruction is enough
# or if we need few-shot examples in v3 to enforce it fully.

extraction_prompt_v2 = """
Task: Extract specific data points from the provided customer feedback
into a structured format.

Fields to Extract:
1. order_id: The numerical identifier for the purchase.
2. order_date: The date of the order (formatted as YYYY-MM-DD).
3. sentiment: Overall sentiment (Positive/Neutral/Negative).
4. reported_issues: List any problems mentioned.

Output Format:
Return the data as a clean JSON object.
Do not include any introductory text or markdown blocks like ```json.

Customer Feedback: "I ordered item #12345 on March 15th.
The delivery was fast but the packaging was damaged."
"""

system_message = "You are a data extraction engine. You only return clean JSON. No markdown, no explanation, no commentary."
temperature = 0.7
num_runs = 15

responses_data = []

print("Starting 15-run test: Data Extraction v2")
print(f"System message: '{system_message}'\n")
print("=" * 80)

for i in range(num_runs):
    print(f"Executing Run {i+1}/{num_runs}...")
    response = call_openai(
        extraction_prompt_v2,
        system_message=system_message,
        temperature=temperature,
        max_tokens=MAX_TOKENS_EXTRACTION
    )

    if is_error_response(response):
        print(f"API Error: {response}")
        responses_data.append({
            "run_id": i + 1,
            "response": response,
            "is_success": False,
            "has_fields": False,
            "is_parseable": False,
            "failure_pattern": "API Error"
        })
        continue

    # Check 1: All required fields present
    required_fields = ['order_id', 'order_date', 'sentiment', 'reported_issues']
    has_fields = all(field in response.lower() for field in required_fields)

    # Check 2: Clean parseable JSON — no markdown wrappers
    clean_response = response.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        json.loads(clean_response)
        is_parseable = True
    except json.JSONDecodeError:
        is_parseable = False

    is_valid = has_fields and is_parseable

    if not has_fields:
        failure_pattern = "Missing Fields"
    elif not is_parseable:
        failure_pattern = "Markdown Wrapper -- not clean JSON"
    else:
        failure_pattern = "None (Success)"

    responses_data.append({
        "run_id": i + 1,
        "response": response,
        "is_success": is_valid,
        "has_fields": has_fields,
        "is_parseable": is_parseable,
        "failure_pattern": failure_pattern
    })

    print(f"Response: '{response[:80]}...'")
    print(f"Has fields: {'yes' if has_fields else 'no'} | Parseable JSON: {'yes' if is_parseable else 'no'}")
    print(f"Valid: {'yes' if is_valid else 'no'} -- {failure_pattern}")
    print("-" * 40)
    time.sleep(0.5)

df_extraction_v2 = pd.DataFrame(responses_data)

# Compare v1 vs v2
v1_rate = df_extraction_v1['is_success'].sum() / len(df_extraction_v1) * 100
v2_rate = df_extraction_v2['is_success'].sum() / len(df_extraction_v2) * 100

print("\n" + "=" * 60)
print("COMPARISON: Data Extraction v1 vs v2")
print("=" * 60)
print(f"v1 Success Rate: {v1_rate:.1f}%")
print(f"v2 Success Rate: {v2_rate:.1f}%")
print(f"Improvement:     +{v2_rate - v1_rate:.1f}%")
print(f"\nFailure Patterns v2:")
print(df_extraction_v2['failure_pattern'].value_counts().to_string())

Starting 15-run test: Data Extraction v2
System message: 'You are a data extraction engine. You only return clean JSON. No markdown, no explanation, no commentary.'

Executing Run 1/15...
Response: '{
  "order_id": 12345,
  "order_date": "2023-03-15",
  "sentiment": "Neutral",
 ...'
Has fields: yes | Parseable JSON: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 2/15...
Response: '{
  "order_id": 12345,
  "order_date": "2023-03-15",
  "sentiment": "Neutral",
 ...'
Has fields: yes | Parseable JSON: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 3/15...
Response: '{
  "order_id": 12345,
  "order_date": "2023-03-15",
  "sentiment": "Negative",
...'
Has fields: yes | Parseable JSON: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 4/15...
Response: '{
  "order_id": 12345,
  "order_date": "2023-03-15",
  "sentiment": "Neutral",
 ...'
Has fields: yes | Parseable JSON: yes


In [31]:
# ============================================================
# Cell 21 — Task 1: Sentiment Analysis v3 (Few-Shot)
# ============================================================

# WHAT CHANGED FROM v2:
# v2 told the model what to do — use one word, pick from these labels.
# v3 SHOWS the model exactly what we want through concrete examples.
# The examples eliminate any ambiguity about format, capitalisation,
# and punctuation that v2 instructions alone couldn't fully enforce.

sentiment_prompt_v3 = """
Classify the sentiment of customer messages.
Respond with exactly one word: Positive, Negative, or Neutral.

Examples:
Message: "This product is amazing, best purchase I've made!"
Label: Positive

Message: "It broke after one day. Complete waste of money."
Label: Negative

Message: "The item arrived on time and was as described."
Label: Neutral

Message: "Terrible experience, I want a refund immediately."
Label: Negative

Now classify this:
Message: "I love this product! It's exactly what I needed."
Label:"""

system_message = "You are a sentiment classification engine. You only respond with a single word: Positive, Negative, or Neutral. Nothing else."
temperature = 0.7
num_runs = 15

responses_data = []

print("Starting 15-run test: Sentiment Analysis v3")
print("Prompt version: v3 (few-shot examples)\n")
print("=" * 80)

for i in range(num_runs):
    print(f"Executing Run {i+1}/{num_runs}...")
    response = call_openai(
        sentiment_prompt_v3,
        system_message=system_message,
        temperature=temperature,
        max_tokens=MAX_TOKENS_SENTIMENT
    )

    if is_error_response(response):
        print(f"API Error: {response}")
        responses_data.append({
            "run_id": i + 1,
            "response": response,
            "is_success": False,
            "failure_pattern": "API Error"
        })
        continue

    valid_labels = ["positive", "negative", "neutral"]
    is_valid = response.strip().lower() in valid_labels
    failure_pattern = "None (Success)" if is_valid else "Wrong Format"

    responses_data.append({
        "run_id": i + 1,
        "response": response,
        "is_success": is_valid,
        "failure_pattern": failure_pattern
    })

    print(f"Response: '{response}'")
    print(f"Valid: {'yes' if is_valid else 'no'} -- {failure_pattern}")
    print("-" * 40)
    time.sleep(0.5)

df_sentiment_v3 = pd.DataFrame(responses_data)

# Compare all three versions side by side
v1_rate = df_sentiment_v1['is_success'].sum() / len(df_sentiment_v1) * 100
v2_rate = df_sentiment_v2['is_success'].sum() / len(df_sentiment_v2) * 100
v3_rate = df_sentiment_v3['is_success'].sum() / len(df_sentiment_v3) * 100

print("\n" + "=" * 60)
print("COMPARISON: Sentiment Analysis v1 vs v2 vs v3")
print("=" * 60)
print(f"v1 Success Rate: {v1_rate:.1f}%  (zero-shot, no instructions)")
print(f"v2 Success Rate: {v2_rate:.1f}%  (structured instructions)")
print(f"v3 Success Rate: {v3_rate:.1f}%  (few-shot examples)")
print(f"\nTotal improvement v1 to v3: +{v3_rate - v1_rate:.1f}%")
print(f"\nFailure Patterns v3:")
print(df_sentiment_v3['failure_pattern'].value_counts().to_string())

Starting 15-run test: Sentiment Analysis v3
Prompt version: v3 (few-shot examples)

Executing Run 1/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 2/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 3/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 4/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 5/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 6/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 7/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Run 8/15...
Response: 'Positive'
Valid: yes -- None (Success)
----------------------------------------
Executing Ru

In [39]:
# ============================================================
# Cell 22 — Task 3: Data Extraction v3 (Chain-of-Thought)
# ============================================================

# WHAT CHANGED FROM v2:
# v2 told the model what fields to extract and what format to use.
# v3 adds Chain-of-Thought — the model must reason through each field
# explicitly before committing to output.
#
# ROOT CAUSE FIX: Prompt now specifies exact field names with underscores.
# Without this the model returns "Order ID" instead of "order_id"
# which breaks the field name validation.

import re

extraction_prompt_v3 = """
Extract data from this customer feedback into clean JSON.

Think through each field step by step before writing your final answer:

Step 1 - order_id: Find any order or item number mentioned.
Step 2 - order_date: Find the date and convert to YYYY-MM-DD format.
         If the year is not mentioned use 2024.
Step 3 - sentiment: Consider ALL positive and negative signals.
         If mixed, lean Neutral.
Step 4 - reported_issues: List only actual problems, not neutral observations.

After your reasoning write your final output as clean JSON using
these exact field names: order_id, order_date, sentiment, reported_issues.
No markdown blocks, no explanation after the JSON.

Customer Feedback: "I ordered item #12345 on March 15th.
The delivery was fast but the packaging was damaged."
"""

system_message = "You are a data extraction engine. Think step by step then output clean JSON only. No markdown. Use snake_case field names exactly as specified."
temperature = 0.7
num_runs = 15

responses_data = []

print("Starting 15-run test: Data Extraction v3")
print("Prompt version: v3 (Chain-of-Thought)\n")
print("=" * 80)

for i in range(num_runs):
    print(f"Executing Run {i+1}/{num_runs}...")
    response = call_openai(
        extraction_prompt_v3,
        system_message=system_message,
        temperature=temperature,
        max_tokens=300
    )

    if is_error_response(response):
        print(f"API Error: {response}")
        responses_data.append({
            "run_id": i + 1,
            "response": response,
            "is_success": False,
            "has_fields": False,
            "is_parseable": False,
            "failure_pattern": "API Error"
        })
        continue

    # Extract JSON using rfind — gets the LAST { and } in the response
    # This skips all the CoT reasoning text that comes before the JSON
    json_start = response.rfind('{')
    json_end = response.rfind('}')

    if json_start != -1 and json_end != -1:
        json_str = response[json_start:json_end + 1]
    else:
        json_str = response

    # Check 1: All required fields present using exact snake_case names
    required_fields = ['order_id', 'order_date', 'sentiment', 'reported_issues']
    has_fields = all(field in response.lower() for field in required_fields)

    # Check 2: The extracted JSON block is actually parseable
    try:
        json.loads(json_str)
        is_parseable = True
    except json.JSONDecodeError:
        is_parseable = False

    is_valid = has_fields and is_parseable

    if not has_fields:
        failure_pattern = "Missing Fields"
    elif not is_parseable:
        failure_pattern = "JSON Parse Error"
    else:
        failure_pattern = "None (Success)"

    responses_data.append({
        "run_id": i + 1,
        "response": response,
        "is_success": is_valid,
        "has_fields": has_fields,
        "is_parseable": is_parseable,
        "failure_pattern": failure_pattern
    })

    print(f"Response: '{response[:120]}...'")
    print(f"Has fields: {'yes' if has_fields else 'no'} | Parseable: {'yes' if is_parseable else 'no'}")
    print(f"Valid: {'yes' if is_valid else 'no'} -- {failure_pattern}")
    print("-" * 40)
    time.sleep(0.5)

df_extraction_v3 = pd.DataFrame(responses_data)

v1_rate = df_extraction_v1['is_success'].sum() / len(df_extraction_v1) * 100
v2_rate = df_extraction_v2['is_success'].sum() / len(df_extraction_v2) * 100
v3_rate = df_extraction_v3['is_success'].sum() / len(df_extraction_v3) * 100

print("\n" + "=" * 60)
print("COMPARISON: Data Extraction v1 vs v2 vs v3")
print("=" * 60)
print(f"v1 Success Rate: {v1_rate:.1f}%  (zero-shot, no format)")
print(f"v2 Success Rate: {v2_rate:.1f}%  (structured instructions)")
print(f"v3 Success Rate: {v3_rate:.1f}%  (Chain-of-Thought)")
print(f"\nTotal improvement v1 to v3: +{v3_rate - v1_rate:.1f}%")
print(f"\nFailure Patterns v3:")
print(df_extraction_v3['failure_pattern'].value_counts().to_string())

Starting 15-run test: Data Extraction v3
Prompt version: v3 (Chain-of-Thought)

Executing Run 1/15...
Response: '{
  "order_id": "12345",
  "order_date": "2024-03-15",
  "sentiment": "neutral",
  "reported_issues": ["packaging was da...'
Has fields: yes | Parseable: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 2/15...
Response: '{
  "order_id": "12345",
  "order_date": "2024-03-15",
  "sentiment": "neutral",
  "reported_issues": ["packaging was da...'
Has fields: yes | Parseable: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 3/15...
Response: '{
  "order_id": "12345",
  "order_date": "2024-03-15",
  "sentiment": "neutral",
  "reported_issues": ["packaging was da...'
Has fields: yes | Parseable: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 4/15...
Response: '{
  "order_id": "12345",
  "order_date": "2024-03-15",
  "sentiment": "neutral",
  "reported_issues": ["

In [36]:
# ============================================================
# Cell 23 — Task 2: Product Description v3 (Few-Shot + Structure)
# ============================================================

# WHAT CHANGED FROM v2:
# v2 described the structure in words — headline, 2 sentences, 3 bullets.
# v3 SHOWS that structure with a complete worked example.
# The model now has a concrete template to copy rather than
# instructions to interpret.
#
# WHY BOTH FEW-SHOT AND STRUCTURE FOR PRODUCT:
# Generation tasks need examples more than any other task type
# because there are infinite ways to write a product description.
# The example anchors the length, tone, format, and style all at once.

product_prompt_v3 = """
Write a product description following this exact format and length.

Example:
Input: Bluetooth headphones, $49.99
Output:
### Crystal Clear Sound on the Go
Immerse yourself in premium audio without the premium price tag.
At $49.99 these headphones deliver studio quality comfort for everyday use.
- Battery: 30-hour playtime on a single charge
- Connectivity: Bluetooth 5.0 with 33ft range
- Comfort: Memory foam ear cushions for all-day wear

Now write one following the exact same format:
Input: Wireless mouse, $29.99
Output:
"""

system_message = "You are a professional product copywriter. You follow the example format exactly — one headline, two sentences, exactly three bullet points. Never exceed 100 words."
temperature = 0.7
num_runs = 15
TARGET_WORD_COUNT = 100

responses_data = []

print("Starting 15-run test: Product Description v3")
print("Prompt version: v3 (few-shot + structure)\n")
print("=" * 80)

for i in range(num_runs):
    print(f"Executing Run {i+1}/{num_runs}...")
    response = call_openai(
        product_prompt_v3,
        system_message=system_message,
        temperature=temperature,
        max_tokens=MAX_TOKENS_PRODUCT
    )

    if is_error_response(response):
        print(f"API Error: {response}")
        responses_data.append({
            "run_id": i + 1,
            "response": response,
            "is_success": False,
            "word_count": 0,
            "within_word_limit": False,
            "has_headline": False,
            "has_bullets": False,
            "failure_pattern": "API Error"
        })
        continue

    word_count = len(response.split())
    within_word_limit = word_count <= TARGET_WORD_COUNT
    has_headline = "###" in response
    has_bullets = response.count("- ") >= 3

    is_valid = within_word_limit and has_headline and has_bullets

    if not within_word_limit:
        failure_pattern = f"Over word limit ({word_count} words)"
    elif not has_headline:
        failure_pattern = "Missing headline"
    elif not has_bullets:
        failure_pattern = "Missing bullet points"
    else:
        failure_pattern = "None (Success)"

    responses_data.append({
        "run_id": i + 1,
        "response": response,
        "is_success": is_valid,
        "word_count": word_count,
        "within_word_limit": within_word_limit,
        "has_headline": has_headline,
        "has_bullets": has_bullets,
        "failure_pattern": failure_pattern
    })

    print(f"Response: '{response[:80]}...'")
    print(f"Word count: {word_count} | Within limit: {'yes' if within_word_limit else 'no'} | Headline: {'yes' if has_headline else 'no'} | Bullets: {'yes' if has_bullets else 'no'}")
    print(f"Valid: {'yes' if is_valid else 'no'} -- {failure_pattern}")
    print("-" * 40)
    time.sleep(0.5)

df_product_v3 = pd.DataFrame(responses_data)

v1_rate = df_product_v1['is_success'].sum() / len(df_product_v1) * 100
v2_rate = df_product_v2['is_success'].sum() / len(df_product_v2) * 100
v3_rate = df_product_v3['is_success'].sum() / len(df_product_v3) * 100
v1_avg_words = df_product_v1['word_count'].mean()
v2_avg_words = df_product_v2['word_count'].mean()
v3_avg_words = df_product_v3['word_count'].mean()

print("\n" + "=" * 60)
print("COMPARISON: Product Description v1 vs v2 vs v3")
print("=" * 60)
print(f"v1 Success Rate: {v1_rate:.1f}% | Avg words: {v1_avg_words:.0f}  (zero-shot)")
print(f"v2 Success Rate: {v2_rate:.1f}% | Avg words: {v2_avg_words:.0f}  (structured instructions)")
print(f"v3 Success Rate: {v3_rate:.1f}% | Avg words: {v3_avg_words:.0f}  (few-shot + structure)")
print(f"\nTotal improvement v1 to v3: +{v3_rate - v1_rate:.1f}%")
print(f"\nFailure Patterns v3:")
print(df_product_v3['failure_pattern'].value_counts().to_string())

Starting 15-run test: Product Description v3
Prompt version: v3 (few-shot + structure)

Executing Run 1/15...
Response: '### Effortless Control at Your Fingertips  
Experience seamless navigation with ...'
Word count: 62 | Within limit: yes | Headline: yes | Bullets: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 2/15...
Response: '### Effortless Precision at Your Fingertips  
Experience seamless navigation wit...'
Word count: 55 | Within limit: yes | Headline: yes | Bullets: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 3/15...
Response: '### Effortless Navigation at Your Fingertips  
Experience seamless control with ...'
Word count: 58 | Within limit: yes | Headline: yes | Bullets: yes
Valid: yes -- None (Success)
----------------------------------------
Executing Run 4/15...
Response: '### Precision at Your Fingertips  
Experience seamless navigation with our ergon...'
Word count: 55 | Within limit: y

In [45]:
# ============================================================
# Cell 24 — Final Comparison: All Three Versions
# ============================================================

print("=" * 60)
print("FINAL REPORT: Prompt Engineering Lab")
print("=" * 60)

# ---- Sentiment Analysis ----
v1_sentiment = df_sentiment_v1['is_success'].sum() / len(df_sentiment_v1) * 100
v2_sentiment = df_sentiment_v2['is_success'].sum() / len(df_sentiment_v2) * 100
v3_sentiment = df_sentiment_v3['is_success'].sum() / len(df_sentiment_v3) * 100

# Unique responses shows consistency beyond binary pass/fail
v1_sentiment_unique = df_sentiment_v1['response'].nunique()
v2_sentiment_unique = df_sentiment_v2['response'].nunique()
v3_sentiment_unique = df_sentiment_v3['response'].nunique()

print("\nSENTIMENT ANALYSIS")
print("-" * 40)
print(f"v1 (zero-shot):    {v1_sentiment:.1f}% | {v1_sentiment_unique} unique responses out of 15")
print(f"v2 (structured):   {v2_sentiment:.1f}% | {v2_sentiment_unique} unique responses out of 15")
print(f"v3 (few-shot):     {v3_sentiment:.1f}% | {v3_sentiment_unique} unique responses out of 15")
print(f"Key insight: v1 had {v1_sentiment_unique} different responses — model was guessing format")
print(f"             v3 converged to {v3_sentiment_unique} unique response — perfect consistency")

# ---- Product Description ----
v1_product = df_product_v1['is_success'].sum() / len(df_product_v1) * 100
v2_product = df_product_v2['is_success'].sum() / len(df_product_v2) * 100
v3_product = df_product_v3['is_success'].sum() / len(df_product_v3) * 100
v1_words = df_product_v1['word_count'].mean()
v2_words = df_product_v2['word_count'].mean()
v3_words = df_product_v3['word_count'].mean()
v1_word_std = df_product_v1['word_count'].std()
v2_word_std = df_product_v2['word_count'].std()
v3_word_std = df_product_v3['word_count'].std()

print("\nPRODUCT DESCRIPTION")
print("-" * 40)
print(f"v1 (zero-shot):    {v1_product:.1f}% | avg {v1_words:.0f} words | std dev {v1_word_std:.0f} words")
print(f"v2 (structured):   {v2_product:.1f}% | avg {v2_words:.0f} words | std dev {v2_word_std:.0f} words")
print(f"v3 (few-shot):     {v3_product:.1f}% | avg {v3_words:.0f} words | std dev {v3_word_std:.0f} words")
print(f"Key insight: word count std dev dropped from {v1_word_std:.0f} to {v3_word_std:.0f}")
print(f"             that means responses became {v1_word_std/v3_word_std:.1f}x more consistent in length")

# ---- Data Extraction ----
v1_extraction = df_extraction_v1['is_success'].sum() / len(df_extraction_v1) * 100
v2_extraction = df_extraction_v2['is_success'].sum() / len(df_extraction_v2) * 100
v3_extraction = df_extraction_v3['is_success'].sum() / len(df_extraction_v3) * 100

v1_markdown = (df_extraction_v1['failure_pattern'] == 'Markdown Wrapper - not clean JSON').sum()
v2_markdown = (df_extraction_v2['failure_pattern'] == 'Markdown Wrapper - not clean JSON').sum()
v3_markdown = (df_extraction_v3['failure_pattern'] == 'Markdown Wrapper - not clean JSON').sum()

print("\nDATA EXTRACTION")
print("-" * 40)
print(f"v1 (zero-shot):    {v1_extraction:.1f}% | markdown wrappers: {v1_markdown}/15 runs")
print(f"v2 (structured):   {v2_extraction:.1f}% | markdown wrappers: {v2_markdown}/15 runs")
print(f"v3 (CoT):          {v3_extraction:.1f}% | markdown wrappers: {v3_markdown}/15 runs")
print(f"Key insight: v1 failed purely due to markdown wrappers, not wrong data")
print(f"             v2/v3 eliminated markdown completely with role alignment")

print("\n" + "=" * 60)
print("KEY TAKEAWAYS")
print("=" * 60)
print("1. Binary pass/fail hides the real story — look at secondary metrics")
print("2. v1 failures were format failures, not content failures")
print("3. Role alignment in system_message fixed most issues immediately")
print("4. Word count std dev shows consistency improvement better than pass/fail")
print("5. v3 few-shot examples reduced unique responses to 1 — perfect consistency")
print("6. max_tokens is hard enforcement — prompts alone are not enough")

FINAL REPORT: Prompt Engineering Lab

SENTIMENT ANALYSIS
----------------------------------------
v1 (zero-shot):    0.0% | 10 unique responses out of 15
v2 (structured):   100.0% | 1 unique responses out of 15
v3 (few-shot):     100.0% | 1 unique responses out of 15
Key insight: v1 had 10 different responses — model was guessing format
             v3 converged to 1 unique response — perfect consistency

PRODUCT DESCRIPTION
----------------------------------------
v1 (zero-shot):    0.0% | avg 292 words | std dev 25 words
v2 (structured):   100.0% | avg 80 words | std dev 4 words
v3 (few-shot):     100.0% | avg 57 words | std dev 3 words
Key insight: word count std dev dropped from 25 to 3
             that means responses became 8.7x more consistent in length

DATA EXTRACTION
----------------------------------------
v1 (zero-shot):    0.0% | markdown wrappers: 15/15 runs
v2 (structured):   100.0% | markdown wrappers: 0/15 runs
v3 (CoT):          100.0% | markdown wrappers: 0/15 runs


In [46]:
# ============================================================
# Cell 25 — Part 5: Task Variations
# ============================================================

# WHY: We've proven our v3 prompts work on the original test cases.
# Now we test them on DIFFERENT inputs to see if they generalise
# or if they were overfit to the specific examples we used.
#
# A prompt that only works on one input is not production ready.
# A prompt that works on variations is.

# --- Sentiment Variations ---
sentiment_variations = [
    "The product stopped working after 2 days. Very disappointed.",  # Clear negative
    "It's okay I guess, nothing special.",                           # Weak neutral
    "Absolutely love it, best thing I've bought all year!",          # Strong positive
    "Fast shipping but the product itself is mediocre.",             # Mixed — tricky
    "I neither like nor dislike this product."                       # Explicit neutral
]

# --- Product Description Variations ---
product_variations = [
    ("mechanical keyboard", "$79.99"),
    ("portable phone charger", "$19.99"),
    ("webcam", "$49.99")
]

# --- Extraction Variations ---
extraction_variations = [
    # Missing order ID
    "I placed an order last Tuesday. Delivery took two weeks and the item was broken.",
    # Multiple issues
    "Order #99887 from January 3rd. Wrong item sent, packaging destroyed, and no invoice included.",
    # Only positive feedback
    "Got my order #55432 on March 1st. Everything was perfect, fast delivery and great packaging."
]

print("=" * 60)
print("PART 5: Testing v3 Prompts on Variations")
print("=" * 60)

# ---- Test Sentiment Variations ----
print("\nSENTIMENT VARIATIONS")
print("-" * 40)

for msg in sentiment_variations:
    # Swap the customer message in the v3 prompt
    varied_prompt = f"""
Classify the sentiment of customer messages.
Respond with exactly one word: Positive, Negative, or Neutral.

Examples:
Message: "This product is amazing, best purchase I've made!"
Label: Positive

Message: "It broke after one day. Complete waste of money."
Label: Negative

Message: "The item arrived on time and was as described."
Label: Neutral

Message: "Terrible experience, I want a refund immediately."
Label: Negative

Now classify this:
Message: "{msg}"
Label:"""

    result = call_openai(
        varied_prompt,
        system_message="You are a sentiment classification engine. You only respond with a single word: Positive, Negative, or Neutral. Nothing else.",
        temperature=0.7,
        max_tokens=MAX_TOKENS_SENTIMENT
    )

    valid_labels = ["positive", "negative", "neutral"]
    is_valid = result.strip().lower() in valid_labels
    print(f"Input:  '{msg[:60]}...' " if len(msg) > 60 else f"Input:  '{msg}'")
    print(f"Output: '{result}' | Valid: {'yes' if is_valid else 'no'}")
    print()
    time.sleep(0.5)

# ---- Test Product Variations ----
print("\nPRODUCT DESCRIPTION VARIATIONS")
print("-" * 40)

for product, price in product_variations:
    varied_prompt = f"""
Write a product description following this exact format and length.

Example:
Input: Bluetooth headphones, $49.99
Output:
### Crystal Clear Sound on the Go
Immerse yourself in premium audio without the premium price tag.
At $49.99 these headphones deliver studio quality comfort for everyday use.
- Battery: 30-hour playtime on a single charge
- Connectivity: Bluetooth 5.0 with 33ft range
- Comfort: Memory foam ear cushions for all-day wear

Now write one following the exact same format:
Input: {product}, {price}
Output:
"""

    result = call_openai(
        varied_prompt,
        system_message="You are a professional product copywriter. You follow the example format exactly — one headline, two sentences, exactly three bullet points. Never exceed 100 words.",
        temperature=0.7,
        max_tokens=MAX_TOKENS_PRODUCT
    )

    word_count = len(result.split())
    has_headline = "###" in result
    has_bullets = result.count("- ") >= 3
    within_limit = word_count <= 100

    print(f"Input:  {product}, {price}")
    print(f"Output: '{result[:80]}...'")
    print(f"Words: {word_count} | Headline: {'yes' if has_headline else 'no'} | Bullets: {'yes' if has_bullets else 'no'} | Within limit: {'yes' if within_limit else 'no'}")
    print()
    time.sleep(0.5)

# ---- Test Extraction Variations ----
print("\nDATA EXTRACTION VARIATIONS")
print("-" * 40)

for feedback in extraction_variations:
    varied_prompt = f"""
Extract data from this customer feedback into clean JSON.

Think through each field step by step before writing your final answer:

Step 1 - order_id: Find any order or item number mentioned. If none, use null.
Step 2 - order_date: Find the date and convert to YYYY-MM-DD format.
         If the year is not mentioned use 2024. If no date, use null.
Step 3 - sentiment: Consider ALL positive and negative signals.
         If mixed, lean Neutral.
Step 4 - reported_issues: List only actual problems. If none, use empty list.

After your reasoning write your final output as clean JSON using
these exact field names: order_id, order_date, sentiment, reported_issues.
No markdown blocks, no explanation after the JSON.

Customer Feedback: "{feedback}"
"""

    result = call_openai(
        varied_prompt,
        system_message="You are a data extraction engine. Think step by step then output clean JSON only. No markdown. Use snake_case field names exactly as specified.",
        temperature=0.7,
        max_tokens=300
    )

    # Extract JSON from response
    json_start = result.rfind('{')
    json_end = result.rfind('}')
    json_str = result[json_start:json_end + 1] if json_start != -1 else result

    try:
        parsed = json.loads(json_str)
        is_parseable = True
    except json.JSONDecodeError:
        is_parseable = False

    required_fields = ['order_id', 'order_date', 'sentiment', 'reported_issues']
    has_fields = all(field in result.lower() for field in required_fields)

    print(f"Input:     '{feedback[:70]}...' " if len(feedback) > 70 else f"Input: '{feedback}'")
    print(f"Output:    '{result[:100]}...'")
    print(f"Parseable: {'yes' if is_parseable else 'no'} | Has fields: {'yes' if has_fields else 'no'}")
    print()
    time.sleep(0.5)

PART 5: Testing v3 Prompts on Variations

SENTIMENT VARIATIONS
----------------------------------------
Input:  'The product stopped working after 2 days. Very disappointed.'
Output: 'Negative' | Valid: yes

Input:  'It's okay I guess, nothing special.'
Output: 'Neutral' | Valid: yes

Input:  'Absolutely love it, best thing I've bought all year!'
Output: 'Positive' | Valid: yes

Input:  'Fast shipping but the product itself is mediocre.'
Output: 'Neutral' | Valid: yes

Input:  'I neither like nor dislike this product.'
Output: 'Neutral' | Valid: yes


PRODUCT DESCRIPTION VARIATIONS
----------------------------------------
Input:  mechanical keyboard, $79.99
Output: '### Elevate Your Typing Experience  
Transform your workspace with our precision...'
Words: 50 | Headline: yes | Bullets: yes | Within limit: yes

Input:  portable phone charger, $19.99
Output: '### Power Up Anywhere, Anytime  
Stay connected with this compact portable phone...'
Words: 55 | Headline: yes | Bullets: yes | Wi

In [47]:
# ============================================================
# Cell 26 — Final Evaluation and Comparison (Step 13)
# ============================================================

# WHY: This is the definitive evaluation cell.
# We run all three v3 prompts 15 times on the original test cases
# and build one comprehensive comparison table covering all versions.
# This is your primary submission evidence.

print("=" * 60)
print("STEP 13: FINAL EVALUATION AND COMPARISON")
print("=" * 60)

# ---- Sentiment Summary Table ----
print("\nSENTIMENT ANALYSIS — FULL COMPARISON")
print("-" * 60)
print(f"{'Version':<20} {'Success Rate':>12} {'Unique Responses':>18} {'Technique'}")
print("-" * 60)

v1_s_rate = df_sentiment_v1['is_success'].sum() / len(df_sentiment_v1) * 100
v2_s_rate = df_sentiment_v2['is_success'].sum() / len(df_sentiment_v2) * 100
v3_s_rate = df_sentiment_v3['is_success'].sum() / len(df_sentiment_v3) * 100
v1_s_unique = df_sentiment_v1['response'].nunique()
v2_s_unique = df_sentiment_v2['response'].nunique()
v3_s_unique = df_sentiment_v3['response'].nunique()

print(f"{'v1 (zero-shot)':<20} {v1_s_rate:>11.1f}% {v1_s_unique:>17} {'No instructions'}")
print(f"{'v2 (structured)':<20} {v2_s_rate:>11.1f}% {v2_s_unique:>17} {'Role + labels + max_tokens'}")
print(f"{'v3 (few-shot)':<20} {v3_s_rate:>11.1f}% {v3_s_unique:>17} {'+ Few-shot examples'}")
print(f"\nImprovement v1 to v3: +{v3_s_rate - v1_s_rate:.1f}%")
print(f"Consistency gain:     {v1_s_unique} unique responses down to {v3_s_unique}")

# ---- Product Summary Table ----
print("\nPRODUCT DESCRIPTION — FULL COMPARISON")
print("-" * 60)
print(f"{'Version':<20} {'Success Rate':>12} {'Avg Words':>10} {'Std Dev':>8} {'Technique'}")
print("-" * 60)

v1_p_rate = df_product_v1['is_success'].sum() / len(df_product_v1) * 100
v2_p_rate = df_product_v2['is_success'].sum() / len(df_product_v2) * 100
v3_p_rate = df_product_v3['is_success'].sum() / len(df_product_v3) * 100
v1_p_words = df_product_v1['word_count'].mean()
v2_p_words = df_product_v2['word_count'].mean()
v3_p_words = df_product_v3['word_count'].mean()
v1_p_std = df_product_v1['word_count'].std()
v2_p_std = df_product_v2['word_count'].std()
v3_p_std = df_product_v3['word_count'].std()

print(f"{'v1 (zero-shot)':<20} {v1_p_rate:>11.1f}% {v1_p_words:>9.0f} {v1_p_std:>7.0f}  {'No instructions'}")
print(f"{'v2 (structured)':<20} {v2_p_rate:>11.1f}% {v2_p_words:>9.0f} {v2_p_std:>7.0f}  {'Role + structure + max_tokens'}")
print(f"{'v3 (few-shot)':<20} {v3_p_rate:>11.1f}% {v3_p_words:>9.0f} {v3_p_std:>7.0f}  {'+ Few-shot example'}")
print(f"\nImprovement v1 to v3: +{v3_p_rate - v1_p_rate:.1f}%")
print(f"Word count reduction: {v1_p_words:.0f} words down to {v3_p_words:.0f} words")
print(f"Consistency gain:     std dev {v1_p_std:.0f} down to {v3_p_std:.0f}")

# ---- Extraction Summary Table ----
print("\nDATA EXTRACTION — FULL COMPARISON")
print("-" * 60)
print(f"{'Version':<20} {'Success Rate':>12} {'Markdown Fails':>15} {'Technique'}")
print("-" * 60)

v1_e_rate = df_extraction_v1['is_success'].sum() / len(df_extraction_v1) * 100
v2_e_rate = df_extraction_v2['is_success'].sum() / len(df_extraction_v2) * 100
v3_e_rate = df_extraction_v3['is_success'].sum() / len(df_extraction_v3) * 100

v1_md = (df_extraction_v1['failure_pattern'] == 'Markdown Wrapper - not clean JSON').sum()
v2_md = (df_extraction_v2['failure_pattern'] == 'Markdown Wrapper - not clean JSON').sum()
v3_md = (df_extraction_v3['failure_pattern'] == 'Markdown Wrapper - not clean JSON').sum()

print(f"{'v1 (zero-shot)':<20} {v1_e_rate:>11.1f}% {v1_md:>14}  {'No format specified'}")
print(f"{'v2 (structured)':<20} {v2_e_rate:>11.1f}% {v2_md:>14}  {'Role + fields + no markdown'}")
print(f"{'v3 (CoT)':<20} {v3_e_rate:>11.1f}% {v3_md:>14}  {'+ Chain-of-Thought reasoning'}")
print(f"\nImprovement v1 to v3: +{v3_e_rate - v1_e_rate:.1f}%")
print(f"Markdown failures:    {v1_md}/15 down to {v3_md}/15")

# ---- Overall Summary ----
print("\n" + "=" * 60)
print("OVERALL IMPROVEMENT SUMMARY")
print("=" * 60)
print(f"Sentiment:   {v1_s_rate:.0f}% -> {v2_s_rate:.0f}% -> {v3_s_rate:.0f}%")
print(f"Product:     {v1_p_rate:.0f}% -> {v2_p_rate:.0f}% -> {v3_p_rate:.0f}% (words: {v1_p_words:.0f} -> {v2_p_words:.0f} -> {v3_p_words:.0f})")
print(f"Extraction:  {v1_e_rate:.0f}% -> {v2_e_rate:.0f}% -> {v3_e_rate:.0f}%")

# ---- Lab Time Report ----
LAB_END = datetime.now()
total_time = LAB_END - LAB_START
minutes = int(total_time.total_seconds() // 60)
seconds = int(total_time.total_seconds() % 60)

print("\n" + "=" * 60)
print("LAB TIME REPORT")
print("=" * 60)
print(f"Start time:  {LAB_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"End time:    {LAB_END.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total time:  {minutes} minutes {seconds} seconds")

STEP 13: FINAL EVALUATION AND COMPARISON

SENTIMENT ANALYSIS — FULL COMPARISON
------------------------------------------------------------
Version              Success Rate   Unique Responses Technique
------------------------------------------------------------
v1 (zero-shot)               0.0%                10 No instructions
v2 (structured)            100.0%                 1 Role + labels + max_tokens
v3 (few-shot)              100.0%                 1 + Few-shot examples

Improvement v1 to v3: +100.0%
Consistency gain:     10 unique responses down to 1

PRODUCT DESCRIPTION — FULL COMPARISON
------------------------------------------------------------
Version              Success Rate  Avg Words  Std Dev Technique
------------------------------------------------------------
v1 (zero-shot)               0.0%       292      25  No instructions
v2 (structured)            100.0%        80       4  Role + structure + max_tokens
v3 (few-shot)              100.0%        57       3  + Fe